In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np


Instructions for updating:
non-resource variables are not supported in the long term


학습 데이터: [공부시간, 과외시간], 7행 2열

In [2]:
xData = [[2, 3], [4, 3], [6, 4], [8, 6], [10, 7], [12, 8], [14, 9]]
print(type(xData))
print(xData)

xData = np.array(xData)
print(type(xData))
print(xData)

<class 'list'>
[[2, 3], [4, 3], [6, 4], [8, 6], [10, 7], [12, 8], [14, 9]]
<class 'numpy.ndarray'>
[[ 2  3]
 [ 4  3]
 [ 6  4]
 [ 8  6]
 [10  7]
 [12  8]
 [14  9]]


실제값: [합격여부], 1행 7열

In [3]:
yData = [0, 0, 0, 1, 1, 1, 1]
print(type(yData))
print(yData)

yData = np.array(yData)
print(type(yData))
print(yData)

<class 'list'>
[0, 0, 0, 1, 1, 1, 1]
<class 'numpy.ndarray'>
[0 0 0 1 1 1 1]


reshape() 메소드로 yData를 1행 7열 1차원 배열에서 7행 1열 2차원 배열로 변경한다.

In [4]:
yData = yData.reshape(7, 1)
print(yData)

[[0]
 [0]
 [0]
 [1]
 [1]
 [1]
 [1]]


xData와 yData를 저장할 placeholder를 만든다.  
placeholder에 `2차원 이상`의 배열을 넣어주려는 경우 shape 속성을 이용해서 넣어줄 `배열의 차원을 반드시 지정`해야 한다.  
`[None, 2]` 형태로 shape을 지정하면 행의 개수는 `몇개라도 상관없고` 열의 개수는 무조건 2개라는 의미이다.

In [5]:
x = tf.placeholder(dtype=tf.float64, shape=[None, 2]) # xData가 대입될 placeholder를 선언한다.
y = tf.placeholder(dtype=tf.float64, shape=[None, 1]) # yData가 대입될 placeholder를 선언한다.

가중치와 바이어스 값을 임의의 값으로 설정한다.

In [6]:
# 행렬의 내적을 이용해서 각 피처와 가중치의 곱의 합을 계산하기 위해서 가중치를 2행 1열 난수로 발생시킨다.
a = tf.Variable(tf.random_uniform([2, 1], dtype=tf.float64))
b = tf.Variable(tf.random_uniform([1], dtype=tf.float64))

sess = tf.Session()
sess.run(tf.global_variables_initializer())

print(f'a\n{sess.run(a)}\nb\n{sess.run(b)}')
print(f'a1 = {sess.run(a[0][0])}, a2 = {sess.run(a[1][0])}, b = {sess.run(b)[0]}')

a
[[0.46182981]
 [0.98980336]]
b
[0.25613723]
a1 = 0.46182980706622523, a2 = 0.9898033607409249, b = 0.25613723328692317


시그모이드 방정식, 오차 함수, 경사 하강법 수식을 만든다.

In [7]:
# Y = 1 / (1 + np.e ** -(a * xData + b))
# matmul() 메소드로 tensorflow에서 행렬의 내적을 계산한다.
# sigmoid() 메소드로 tensorflow에서 시그모이드 방정식을 계산한다.
Y = tf.sigmoid(tf.matmul(x, a) + b)

# -tf.reduce_mean(실제값 * tf.log(예측값) + (1 - 실제값) * tf.log(1 - 예측값))
loss = -tf.reduce_mean(y * tf.log(Y) + (1 - y) * tf.log(1 - Y))

gradient_descent = tf.train.GradientDescentOptimizer(0.1).minimize(loss)

## 시그모이드 메소드를 실행한 결과로 예측값(1 또는 0)을 계산한다.

시그모이드 메소드의 실행 결과가 0.6 이상이면 1을 0.6 미만이면 0을 리턴시킨다.

cast() 메소드로 tensorflow에서 데이터 형 변환을 할 수 있다.  
`case(형 변환 할 데이터, 형 변환할 데이터 타입)`

In [8]:
predict = tf.cast(tf.constant([1.9, 2.3]), dtype=tf.int32)
print(sess.run(predict))

[1 2]


cast() 메소드는 형 변환할 데이터가 boolean 타입일 경우 True는 1로 False는 0으로 형 변환한다.

In [9]:
predict = tf.cast(tf.constant([True, False]), dtype=tf.float64)
print(sess.run(predict))
predict = tf.cast(tf.constant([0.6 >= 0.6, 0.6 < 0.6]), dtype=tf.float64)
print(sess.run(predict))

[1. 0.]
[1. 0.]


시그모이드 메소드가 예측한 실제값(실수)을 cast() 메소드를 이용해서 0 또는 1로 변환한다. => 최종 예측값

In [10]:
predict = tf.cast(Y >= 0.6, dtype=tf.float64)

최종 예측값(predict)과 실제값(y)이 일치하는 정도(정확도)를 계산한다.

equal() 메소드는 tensorflow에서 인수로 지정한 두 값이 같으면 True, 다르면 False를 리턴한다.

In [11]:
print(sess.run(tf.equal(1, 1)), sess.run(tf.equal(1, 0)))

True False


정확도는 equal() 메소드로 최종 예측값과 실제값이 같은가 비교 후 리턴되는 True 또는 False를 cast() 메소드로 1.0 또는 0.0으로 형 변환하고 평균을 계산하면 된다.

In [12]:
accuracy = tf.reduce_mean(tf.cast(tf.equal(predict, y), dtype=tf.float64))

학습시킨다.

In [20]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

for epoch in range(301):
    _, _loss, _a, _b, _accuracy = sess.run([gradient_descent, loss, a, b, accuracy], feed_dict={x: xData, y: yData})
    if epoch % 20 == 0:
        print(
            'epoch: {:3d}, loss: {:8.4f}, a1: {:8.4f}, a2: {:8.4f}, b: {:8.4f}, accuracy: {:8.2%}'.format(epoch, _loss, _a[0][0], _a[1][0], _b[0], _accuracy)
        )

epoch:   0, loss:   1.8959, a1:   0.2032, a2:   0.5720, b:   0.4992, accuracy:   57.14%
epoch:  20, loss:   0.5892, a1:   0.0505, a2:   0.1097, b:   0.1031, accuracy:   57.14%
epoch:  40, loss:   0.5138, a1:   0.1974, a2:  -0.0634, b:  -0.2064, accuracy:   85.71%
epoch:  60, loss:   0.4612, a1:   0.3170, a2:  -0.1900, b:  -0.4772, accuracy:   85.71%
epoch:  80, loss:   0.4229, a1:   0.4111, a2:  -0.2836, b:  -0.7178, accuracy:   85.71%
epoch: 100, loss:   0.3938, a1:   0.4858, a2:  -0.3527, b:  -0.9350, accuracy:   85.71%
epoch: 120, loss:   0.3708, a1:   0.5458, a2:  -0.4036, b:  -1.1337, accuracy:   85.71%
epoch: 140, loss:   0.3519, a1:   0.5945, a2:  -0.4406, b:  -1.3176, accuracy:   85.71%
epoch: 160, loss:   0.3359, a1:   0.6345, a2:  -0.4671, b:  -1.4893, accuracy:   85.71%
epoch: 180, loss:   0.3221, a1:   0.6674, a2:  -0.4852, b:  -1.6508, accuracy:   85.71%
epoch: 200, loss:   0.3099, a1:   0.6948, a2:  -0.4967, b:  -1.8036, accuracy:  100.00%
epoch: 220, loss:   0.2990, a1: 